# Main Figure Galactic Simple Tutorial

This notebook shows a direct `oviz` workflow for generating the galactic-simple main figure.

It does **not** import `/Users/swiggumc/Desktop/oviz/scripts/main_figure.py` or the external `main_figure.ipynb`. Instead, it uses the public `oviz` API directly:

- `Trace`
- `TraceCollection`
- `Animate3D`

The example below rebuilds the lightweight galactic background view using the same local cluster catalog, the Edenhofer dust cube, and the Milky Way overlay image.

## Imports

Only `oviz`, standard scientific Python packages, and notebook display helpers are used here.

In [29]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import FileLink, IFrame, display

from oviz import Animate3D, Trace, TraceCollection

## Configure local inputs

These are the local data products used by the current galactic-simple figure.

If your files live somewhere else, edit these paths before running the notebook.

In [30]:
HOME = Path.home()

cluster_catalog_path = HOME / "Desktop" / "astro_research" / "supernovae_map_work" / "clusters" / "vels_output" / "2026-02-04" / "cluster_velocities_jan2026.csv"
chronos_age_path = HOME / "Downloads" / "hunt_sample_chronos_ages_multiprocessing_feb_2026.csv"
edenhofer_fits_path = HOME / "Downloads" / "mean_and_std_xyz-2.fits"
galactic_plane_image_path = HOME / "Downloads" / "Top-down_view_of_the_Milky_Way.jpg"

output_html = Path("/tmp/main_figure_simple_galactic_tutorial.html")

required_paths = {
    "cluster_catalog": cluster_catalog_path,
    "chronos_ages": chronos_age_path,
    "edenhofer_fits": edenhofer_fits_path,
    "galactic_plane_image": galactic_plane_image_path,
}

missing = []
for label, path in required_paths.items():
    exists = path.exists()
    print(f"{label:20s} {path} {'OK' if exists else 'MISSING'}")
    if not exists:
        missing.append(label)

if missing:
    raise FileNotFoundError("Missing required local inputs: " + ", ".join(missing))

output_html

cluster_catalog      /Users/swiggumc/Desktop/astro_research/supernovae_map_work/clusters/vels_output/2026-02-04/cluster_velocities_jan2026.csv OK
chronos_ages         /Users/swiggumc/Downloads/hunt_sample_chronos_ages_multiprocessing_feb_2026.csv OK
edenhofer_fits       /Users/swiggumc/Downloads/mean_and_std_xyz-2.fits OK
galactic_plane_image /Users/swiggumc/Downloads/Top-down_view_of_the_Milky_Way.jpg OK


PosixPath('/tmp/main_figure_simple_galactic_tutorial.html')

## Load and prepare the cluster catalog

This reproduces the cluster-catalog preparation used for the main figure, but keeps only the pieces needed for the galactic-simple view:

- `Sun`
- `Clusters (< 60 Myr)`

The red `< 15 Myr` layer and the family/structure subgroups are intentionally omitted here, since the current galactic-simple figure does not use them.

In [31]:
df_hunt_full = pd.read_csv(cluster_catalog_path)
ages_chronos = pd.read_csv(chronos_age_path)

df_hunt_full = pd.merge(
    df_hunt_full,
    ages_chronos[["name", "age_chronos_lo", "age_chronos_mode", "age_chronos_hi"]],
    on="name",
    how="left",
)
df_hunt_full["age_myr"] = df_hunt_full["age_chronos_mode"]

df_hunt_full = df_hunt_full.drop(columns=["x", "y", "z", "U", "V", "W", "U_err", "V_err", "W_err"])
df_hunt_full = df_hunt_full.rename(
    columns={
        "U_2026": "U",
        "V_2026": "V",
        "W_2026": "W",
        "U_err_2026": "U_err",
        "V_err_2026": "V_err",
        "W_err_2026": "W_err",
        "x_new": "x",
        "y_new": "y",
        "z_new": "z",
    }
)

df_hunt_good = df_hunt_full.loc[
    (df_hunt_full["U_err"] < 10)
    & (df_hunt_full["V_err"] < 10)
    & (df_hunt_full["W_err"] < 10)
    & (df_hunt_full["U"].notnull())
    & (df_hunt_full["V"].notnull())
    & (df_hunt_full["W"].notnull())
    & (df_hunt_full["age_myr"].notnull())
    & (df_hunt_full["x"].between(-2000, 2000))
    & (df_hunt_full["y"].between(-2000, 2000))
    & (df_hunt_full["z"].between(-300, 300))
    & (df_hunt_full["n_rvs_2026"] >= 3)
    & (df_hunt_full["class_50"] > 0.5)
    & (df_hunt_full["cst"] > 5)
].copy()

df_hunt_60 = df_hunt_good.loc[df_hunt_good["age_myr"] < 60].copy()

sun = pd.DataFrame(
    {
        "name": ["Sun"],
        "age_myr": [8000.0],
        "U": [0.0],
        "V": [0.0],
        "W": [0.0],
        "x": [0.0],
        "y": [0.0],
        "z": [27.0],
        "n_stars": [1],
    }
)

print(f"Total good clusters: {len(df_hunt_good):,}")
print(f"Clusters < 60 Myr: {len(df_hunt_60):,}")

Total good clusters: 1,193
Clusters < 60 Myr: 469


## Build the `oviz` traces

The current galactic-simple figure uses the Sun plus the broad `< 60 Myr` cluster population.

In [32]:
cluster_trace = Trace(
    df_hunt_60,
    data_name="Clusters (< 60 Myr)",
    min_size=0.0,
    max_size=7.0,
    color="#5fd6ff",
    opacity=0.72,
    marker_style="circle",
    show_tracks=False,
    size_by_n_stars=False,
)

sun_trace = Trace(
    sun,
    data_name="Sun",
    min_size=5.0,
    max_size=5.0,
    color="yellow",
    opacity=1.0,
    marker_style="circle",
    show_tracks=False,
    size_by_n_stars=False,
)

traces = TraceCollection([sun_trace, cluster_trace])
trace_groupings = {
    "Clusters": ["Sun", "Clusters (< 60 Myr)"],
}

traces.get_all_clusters()

[<oviz.traces.Trace at 0x3b5de02d0>, <oviz.traces.Trace at 0x39e3c4810>]

## Configure the dust layer and galactic-simple viewer state

This uses the Edenhofer dust cube only. The supernova volume path is intentionally left out.

The `threejs_initial_state` block is where the minimal galactic presentation mode is activated.

In [33]:
greys_cmap = plt.get_cmap("Greys")

edenhofer_volume = {
    "name": "Edenhofer+2024 Dust",
    "path": str(edenhofer_fits_path),
    "hdu": "MEAN",
    "max_resolution": 100,
    "opacity": 1.0,
    "samples": 24,
    "alpha_coef": 200,
    "vmin": 0.0,
    "vmax": 0.0385,
    "colormap": greys_cmap,
    "only_at_t0": True,
}

threejs_initial_state = {
    "current_group": "Clusters",
    "click_selection_enabled": False,
    "active_volume_key": "volume-0",
    "legend_state": {"volume-0": True},
    "volume_state_by_key": {"volume-0": {"visible": True}},
    "minimal_mode_enabled": True,
    "galactic_simple_mode_enabled": True,
    "galactic_plane_image_path": str(galactic_plane_image_path),
    "galactic_plane_size_pc": 40000.0,
    "galactic_plane_opacity": 0.6,
    "initial_zoom_anchor": {"x": 0.0, "y": 0.0, "z": 0.0},
    "camera": {
        "position": {"x": 3700.0, "y": -6550.0, "z": 4700.0},
        "target": {"x": 3000.0, "y": 0.0, "z": 0.0},
        "up": {"x": 0.0, "y": 0.0, "z": 1.0},
    },
    "global_controls": {
        "camera_fov": 70.0,
        "point_size_scale": 0.88,
        "point_glow_strength": 0.60,
        "fade_in_time_myr": 7.0,
    },
}

threejs_initial_state

{'current_group': 'Clusters',
 'click_selection_enabled': False,
 'active_volume_key': 'volume-0',
 'legend_state': {'volume-0': True},
 'volume_state_by_key': {'volume-0': {'visible': True}},
 'minimal_mode_enabled': True,
 'galactic_simple_mode_enabled': True,
 'galactic_plane_image_path': '/Users/swiggumc/Downloads/Top-down_view_of_the_Milky_Way.jpg',
 'galactic_plane_size_pc': 40000.0,
 'galactic_plane_opacity': 0.6,
 'initial_zoom_anchor': {'x': 0.0, 'y': 0.0, 'z': 0.0},
 'camera': {'position': {'x': 3700.0, 'y': -6550.0, 'z': 4700.0},
  'target': {'x': 3000.0, 'y': 0.0, 'z': 0.0},
  'up': {'x': 0.0, 'y': 0.0, 'z': 1.0}},
 'global_controls': {'camera_fov': 70.0,
  'point_size_scale': 0.88,
  'point_glow_strength': 0.6,
  'fade_in_time_myr': 7.0}}

## Render the galactic-simple figure

This is the core `oviz` call. The important pieces are:

- `renderer="threejs"`
- `threejs_initial_state=threejs_initial_state`
- `volumes=[edenhofer_volume]`

No helper scripts are used.

In [34]:
time_int = np.round(np.arange(0, -66, -1), 1)

plot_3d = Animate3D(
    data_collection=traces,
    xyz_widths=(2000, 2000, 400),
    figure_theme="dark",
    trace_grouping_dict=trace_groupings,
)

fig3d = plot_3d.make_plot(
    time=time_int,
    show=False,
    save_name=str(output_html),
    focus_group=None,
    fade_in_time=8,
    fade_in_and_out=False,
    show_age_kde_inset=False,
    include_spiral_arms=False,
    galactic_mode=True,
    show_galactic_guides=False,
    camera_zoom_factor=5,
    show_gc_line=True,
    show_milky_way_model=False,
    renderer="threejs",
    enable_sky_panel=False,
    volumes=[edenhofer_volume],
    threejs_initial_state=threejs_initial_state,
)

output_html.exists(), output_html

(True, PosixPath('/tmp/main_figure_simple_galactic_tutorial.html'))

In [35]:
display(FileLink(output_html))
IFrame(src=output_html.as_uri(), width="100%", height=720)

/tmp/main_figure_simple_galactic_tutorial.html

## Notes

- This tutorial is intentionally `oviz`-native and leaves out the script/notebook wrapper layer.
- It also leaves out the supernova volume machinery.
- If you want to publish the result, copy the generated HTML to your target website directory after you are happy with the output.
- If you want to tune the look, the easiest levers are `threejs_initial_state["camera"]` and `threejs_initial_state["global_controls"]`.